# Log Analysis
This notebook parses the log files to visualize the validation loss against iterations and time. It also extracts key statistics like peak memory, total training time, and step average.

In [ ]:
import os
import glob
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Find all log files in the 'logs' folder
log_files = sorted(glob.glob('logs/*.txt'))
print(f"Found {len(log_files)} log file(s)\n")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss vs Iterations", "Loss vs Time"))

for log_file in log_files:
    iterations = []
    val_losses = []
    val_times = []
    peak_memory = "Not found"
    total_time = "Not found"
    final_step_avg = "Not found"
    
    with open(log_file, 'r') as f:
        for line in f:
            if 'val_loss:' in line:
                m = re.search(r'step:(\d+)/\d+.*val_loss:([\d\.]+).*train_time:(\d+)ms', line)
                if m:
                    iterations.append(int(m.group(1)))
                    val_losses.append(float(m.group(2)))
                    val_times.append(int(m.group(3)) / 1000.0)
                    
            if line.startswith('step:'):
                m = re.search(r'train_time:(\d+)ms step_avg:([\d\.]+)ms', line)
                if m:
                    total_time = f"{int(m.group(1)) / 1000.0:.2f}s"
                    final_step_avg = f"{m.group(2)}ms"
                    
            elif 'peak memory allocated' in line:
                peak_memory = line.strip()
    
    label = os.path.basename(log_file)[:8]
    print(f"--- {os.path.basename(log_file)} ---")
    print(f"Total Training Time: {total_time}")
    print(f"Final Step Average:  {final_step_avg}")
    print(f"Peak Memory:         {peak_memory}\n")
    
    if iterations:
        # Add Iteration trace
        fig.add_trace(
            go.Scatter(x=iterations, y=val_losses, name=f"{label}", mode='lines+markers', 
                       legendgroup=label, hovertemplate='Step: %{x}<br>Loss: %{y:.4f}'),
            row=1, col=1
        )
        # Add Time trace
        fig.add_trace(
            go.Scatter(x=val_times, y=val_losses, name=f"{label}", mode='lines+markers', 
                       legendgroup=label, showlegend=False, hovertemplate='Time: %{x:.2f}s<br>Loss: %{y:.4f}'),
            row=1, col=2
        )

fig.update_layout(
    title_text="Training Logs Comparison",
    template="plotly_dark",
    height=600,
    legend_title="Run ID",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Iterations", row=1, col=1)
fig.update_yaxes(title_text="Validation Loss", row=1, col=1)
fig.update_xaxes(title_text="Training Time (seconds)", row=1, col=2)
fig.update_yaxes(title_text="Validation Loss", row=1, col=2)

fig.show()
